# Spotify Analytics

In [3]:
import pandas as pd

csv_path = "../data/raw/spotify_tracks.csv"
df = pd.read_csv(csv_path)

# 'track_id' column cast to string and strip of whitespace
df['track_id'] = df['track_id'].astype(str).str.strip()

# Keep only records where the length of 'track_id' is exactly 22 chars
df = df[df['track_id'].str.len() == 22]

print(f"Sanitized dataset rows: {len(df)}")

Sanitized dataset rows: 114000


## Normalize the 'Artists' column

In [4]:
# Extract the 'track_id' and 'artists' columns, and drop null values
artists_raw = df[['track_id', 'artists']].dropna()

# Split the string 'artists' column by the semicolon separator (';')
artists_raw['artists'] = df['artists'].str.split(';')

#Explode the list-valued 'artists' column into distinct rows
artists_exploded = df.explode('artists')

# Strip any leading/trailing spaces from the exploded 'artists' column
artists_exploded['artists'] = artists_exploded['artists'].str.strip()

# Create a unique lookup table of artist names (with no duplicates)
artists_lookup = pd.DataFrame({'artist_name': artists_exploded['artists'].unique()}).dropna()

## Popularity-Based Canonical Deduplication

In [6]:
# Sort 'df' by 'popularity' in descending order
df_sorted = df.sort_values('popularity', ascending=False)

# Drop duplicate rows based on 'track_id', keeping only the 'first' instance (the highest popularity)
tracks_clean = df_sorted.drop_duplicates(subset='track_id', keep='first')

# Extract distinct albums from the dataset to create a lookup table
albums_lookup = df[['album_name']].dropna().drop_duplicates()

# Extract distinct genres from the dataset to create a lookup table
genres_lookup = df[['track_genre']].dropna().drop_duplicates()

df.to_csv("../data/cleaned/spotify_cleaned.csv", index=False)